# Context Management
Normalize current input files, compare them with trusted BigQuery context, inspect relevance, and create approval proposals.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'config').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from dq_agent.config import load_app_config
from dq_agent.context_store import read_context, write_context_proposals
from dq_agent.context_utils import (
    build_context_proposals, build_effective_inputs, configure_workflow_logging,
    normalize_input_context, context_workflow_paths, logged_step, rank_relevant_context,
    validate_normalized_context, write_approval_workbook,
)
from dq_agent.reporting import write_json

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
config = load_app_config(ROOT)
paths = context_workflow_paths(config, RUN_ID)
logger = configure_workflow_logging(paths['log'], config.project.log_level)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'LOAD_CONFIGURATION'):
    print('Context enabled:', config.project.context_store.enabled)
    print('Namespace:', config.project.context_store.namespace)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'NORMALIZE_CONTEXT'):
    normalized = normalize_input_context(config, RUN_ID, logger)
    print(f'Normalized records: {len(normalized)}')
display(normalized[['context_type','subject_key','target_table','publish_to_context','content_hash']])

In [ ]:
with logged_step(logger, paths['checkpoint'], 'CHECK_CONFLICTS'):
    validate_normalized_context(normalized)
    print('No conflicting context keys found.')

In [ ]:
with logged_step(logger, paths['checkpoint'], 'RETRIEVE_RELEVANT_CONTEXT'):
    trusted = read_context(config, logger=logger) if config.project.context_store.enabled else pd.DataFrame()
    relevant = rank_relevant_context(normalized, trusted, config.project.confidence.review)
    print(f'Trusted records: {len(trusted)}; relevant candidates: {len(relevant)}')
display(relevant)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'WRITE_CONTEXT_STAGING'):
    proposals = build_context_proposals(normalized, trusted, RUN_ID)
    if config.project.context_store.enabled:
        staging_counts = write_context_proposals(config, proposals, logger)
    else:
        staging_counts = {'accepted': 0, 'existing': 0, 'rejected': 0, 'offline_proposals': len(proposals)}
    print(staging_counts)
display(proposals[['proposal_id','context_type','subject_key','action','confidence']] if not proposals.empty else proposals)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'CREATE_APPROVAL_FILE'):
    approval_file = None
    if not proposals.empty:
        approval_file = paths['pending'] / f'context_approvals_{RUN_ID}.xlsx'
        write_approval_workbook(approval_file, proposals)
        logger.info('CREATE_APPROVAL_FILE output=%s records=%s', approval_file, len(proposals))
    print('Approval file:', approval_file or 'No publishable changes')

In [ ]:
with logged_step(logger, paths['checkpoint'], 'APPLY_CONTEXT_PRECEDENCE'):
    effective_inputs = build_effective_inputs(normalized, trusted)
    effective_file = paths['output'] / 'effective_context.json'
    write_json(effective_file, effective_inputs)
    print('Effective context:', effective_file)
display(pd.DataFrame(effective_inputs['column_mappings']))

## Manual verification
Set one `publish_to_context` value to `true`, rerun with a stable `RUN_ID`, and confirm that the first run creates one proposal while the second does not duplicate it in BigQuery.